# Phase 4 — Community Detection
**Steps 4.1 – 4.6** | Louvain on full + temporal + issue networks, Sankey diagram, NMI, Girvan-Newman.

**Deliverables:** D1 (India Multi-Alignment Matrix) · D2 (Sankey / alluvial diagram)

In [ ]:
import os, pickle, warnings
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import plotly.graph_objects as go
import community as community_louvain  # python-louvain
from sklearn.metrics import adjusted_mutual_info_score
warnings.filterwarnings('ignore')

matplotlib_style = {'figure.facecolor': '#0d1117', 'axes.facecolor': '#161b22',
                    'axes.edgecolor': '#30363d', 'text.color': 'white',
                    'axes.labelcolor': 'white', 'xtick.color': 'white',
                    'ytick.color': 'white', 'figure.dpi': 150}
plt.rcParams.update(matplotlib_style)

ROOT   = os.path.abspath(os.path.join(os.getcwd(), '..'))
NETS   = os.path.join(ROOT, 'results', 'networks')
PLOTS  = os.path.join(ROOT, 'results', 'plots')
TABLES = os.path.join(ROOT, 'results', 'tables')

def load_graph(name):
    with open(os.path.join(NETS, f'{name}.pkl'), 'rb') as f:
        return pickle.load(f)

ERAS = ['full', 'cold_war', 'post_cw', 'post_9_11', 'recent']
ISSUES = ['me', 'co', 'hr', 'di', 'nu', 'ec']
ISSUE_NAMES = {
    'me': 'Israel/Palestine', 'co': 'Colonization',
    'hr': 'Human Rights', 'di': 'Disarmament',
    'nu': 'Nuclear', 'ec': 'Economic Dev'
}

G_full = load_graph('full')
print(f'Full network: {G_full.number_of_nodes()} nodes, {G_full.number_of_edges()} edges')
India = 'India'
India = 'India'

## Step 4.1 – Louvain on Full Network (10 runs, best Q)

In [ ]:
def run_louvain_best(G, n_runs=10):
    """Run Louvain n_runs times, pick partition with highest modularity."""
    best_Q, best_partition = -1, None
    for seed in range(n_runs):
        partition = community_louvain.best_partition(G, weight='weight', random_state=seed)
        Q = community_louvain.modularity(partition, G, weight='weight')
        if Q > best_Q:
            best_Q = Q
            best_partition = partition
    return best_partition, best_Q

print('Running Louvain on full network (10 runs)...')
full_partition, full_Q = run_louvain_best(G_full, n_runs=10)
n_communities = len(set(full_partition.values()))
print(f'Modularity Q = {full_Q:.4f}')
print(f'Number of communities: {n_communities}')

# Community membership counts
comm_counts = pd.Series(full_partition).value_counts().sort_index()
print('\nCommunity sizes:')
print(comm_counts.to_string())

In [ ]:
# Manual community labeling — inspect members
comm_members = {}
for country, comm_id in full_partition.items():
    comm_members.setdefault(comm_id, []).append(country)

for cid, members in sorted(comm_members.items()):
    print(f'\nCommunity {cid} ({len(members)} members):')
    print('  ', ', '.join(sorted(members)[:20]))
    if len(members) > 20:
        print(f'  ... and {len(members)-20} more')

In [ ]:
# Auto-label communities by dominant region
import os
region_df = pd.read_csv(os.path.join(ROOT, 'data', 'external', 'un_regional_groups.csv'))
region_map = dict(zip(region_df['country'], region_df['region']))

community_labels = {}
for cid, members in comm_members.items():
    regions = [region_map.get(c, 'Unknown') for c in members]
    dominant = pd.Series(regions).value_counts().idxmax()
    community_labels[cid] = f'C{cid}:{dominant}'

print('Auto-labels:', community_labels)

# Save full partition
partition_df = pd.DataFrame([
    {'country': c, 'community': cid, 'community_label': community_labels[cid]}
    for c, cid in full_partition.items()
])
partition_df.to_csv(os.path.join(TABLES, 'p4_louvain_full.csv'), index=False)
print('\nIndia community:')
india_row = partition_df[partition_df['country'] == 'India']
print(india_row)

In [ ]:
# Visualise communities on spring layout
COMM_PALETTE = plt.cm.tab20.colors
node_colors = [COMM_PALETTE[full_partition.get(n, 0) % len(COMM_PALETTE)] for n in G_full.nodes]

pos = nx.spring_layout(G_full, seed=42, k=0.3, iterations=50)
fig, ax = plt.subplots(figsize=(15, 11))
fig.patch.set_facecolor('#0d1117')
ax.set_facecolor('#0d1117')

nx.draw_networkx_edges(G_full, pos, ax=ax, alpha=0.15, width=0.4, edge_color='#8b949e')
nx.draw_networkx_nodes(G_full, pos, ax=ax, node_color=node_colors, node_size=35, alpha=0.95)

key_countries = ['United States of America','China','Russia','India','Brazil','Germany',
                 'South Africa','Nigeria','Saudi Arabia']
labels_dict = {n: n.split()[-1] for n in key_countries if n in G_full}
nx.draw_networkx_labels(G_full, pos, labels=labels_dict, ax=ax, font_size=7, font_color='white')

legend_handles = [mpatches.Patch(color=COMM_PALETTE[cid % len(COMM_PALETTE)], label=lbl)
                  for cid, lbl in sorted(community_labels.items())]
ax.legend(handles=legend_handles, loc='lower left', fontsize=7,
          facecolor='#161b22', edgecolor='#30363d', labelcolor='white')
ax.set_title(f'UNGA Full Network — Louvain Communities (Q={full_Q:.3f})',
             color='white', fontsize=13)
ax.axis('off')
plt.tight_layout()
plt.savefig(os.path.join(PLOTS, 'p4_communities_full.png'),
            bbox_inches='tight', facecolor='#0d1117', dpi=180)
plt.show()

## Step 4.2 – Louvain on 4 Temporal Networks

In [ ]:
era_partitions = {}
era_qs = {}

for era in ERAS:
    G_era = load_graph(era)
    if G_era.number_of_edges() < 5:
        print(f'  {era}: Too few edges ({G_era.number_of_edges()}), skipping Louvain')
        continue
    partition, Q = run_louvain_best(G_era, n_runs=10)
    era_partitions[era] = partition
    era_qs[era] = Q
    n_comm = len(set(partition.values()))
    print(f'{era:15s}: Q={Q:.4f}, communities={n_comm}')
    
    # Save
    pdf = pd.DataFrame([{'country': c, 'community': cid} for c, cid in partition.items()])
    pdf.to_csv(os.path.join(TABLES, f'p4_louvain_{era}.csv'), index=False)

## Step 4.3 – Sankey Alluvial Diagram (D2)

In [ ]:
from scipy.optimize import linear_sum_assignment

def align_partitions(p_ref, p_new):
    """Remap community IDs in p_new to best match p_ref using Jaccard similarity."""
    ref_comms = {}
    for c, cid in p_ref.items():
        ref_comms.setdefault(cid, set()).add(c)
    new_comms = {}
    for c, cid in p_new.items():
        new_comms.setdefault(cid, set()).add(c)
    
    ref_ids = sorted(ref_comms)
    new_ids = sorted(new_comms)
    cost = np.zeros((len(ref_ids), len(new_ids)))
    for i, rid in enumerate(ref_ids):
        for j, nid in enumerate(new_ids):
            inter = len(ref_comms[rid] & new_comms[nid])
            union = len(ref_comms[rid] | new_comms[nid])
            cost[i, j] = 1 - (inter / union if union > 0 else 0)
    row_ind, col_ind = linear_sum_assignment(cost)
    remap = {new_ids[col_ind[i]]: ref_ids[row_ind[i]] for i in range(len(row_ind))}
    return {c: remap.get(cid, cid) for c, cid in p_new.items()}

eras_ordered = ['cold_war', 'post_cw', 'post_9_11', 'recent']
aligned_partitions = {}
for i, era in enumerate(eras_ordered):
    if i == 0:
        aligned_partitions[era] = era_partitions.get(era, {})
    else:
        prev = eras_ordered[i-1]
        if era in era_partitions and prev in aligned_partitions:
            aligned_partitions[era] = align_partitions(
                aligned_partitions[prev], era_partitions[era])
        else:
            aligned_partitions[era] = era_partitions.get(era, {})

# Build Sankey data using aligned_partitions
valid_eras = [e for e in eras_ordered if e in aligned_partitions]
era_labels = ['Cold War\n1946–91', 'Post-CW\n1991–01', 'Post-9/11\n2001–14', 'Recent\n2014–15']

if len(valid_eras) >= 2:
    all_countries = set()
    for e in valid_eras:
        all_countries |= set(aligned_partitions[e].keys())
    common_countries = all_countries.copy()
    for e in valid_eras:
        common_countries &= set(aligned_partitions[e].keys())

    node_labels, node_colors_sankey, node_indices = [], [], {}
    COMM_COLORS_HEX = ['#58a6ff','#f85149','#3fb950','#d2a8ff','#ffa657','#79c0ff','#ffab70','#a5d6ff','#ffa8a8','#56d364']
    
    def hex_to_rgba(hex_str, alpha=0.5):
        hex_str = hex_str.lstrip('#')
        r, g, b = tuple(int(hex_str[i:i+2], 16) for i in (0, 2, 4))
        return f'rgba({r}, {g}, {b}, {alpha})'

    for era_idx, era in enumerate(valid_eras):
        p = aligned_partitions[era]
        comms_in_era = sorted(set(p[c] for c in common_countries if c in p))
        for comm in comms_in_era:
            key = (era_idx, comm)
            node_idx = len(node_labels)
            node_indices[key] = node_idx
            n_members = sum(1 for c in common_countries if p[c] == comm)
            node_labels.append(f'{era_labels[era_idx]}\nC{comm}({n_members})')
            node_colors_sankey.append(COMM_COLORS_HEX[comm % len(COMM_COLORS_HEX)])

    sources, targets, values, link_colors = [], [], [], []
    for t in range(len(valid_eras)-1):
        p_src = aligned_partitions[valid_eras[t]]
        p_tgt = aligned_partitions[valid_eras[t+1]]
        for c in common_countries:
            src_key = (t, p_src[c])
            tgt_key = (t+1, p_tgt[c])
            if src_key in node_indices and tgt_key in node_indices:
                sources.append(node_indices[src_key])
                targets.append(node_indices[tgt_key])
                values.append(1)
                link_colors.append(hex_to_rgba(COMM_COLORS_HEX[p_src[c] % len(COMM_COLORS_HEX)], 0.4))

    fig_sankey = go.Figure(go.Sankey(
        node=dict(pad=20, thickness=20, label=node_labels, color=node_colors_sankey),
        link=dict(source=sources, target=targets, value=values, color=link_colors)
    ))
    fig_sankey.update_layout(title='UNGA Voting Bloc Evolution (Aligned)', paper_bgcolor='#0d1117', font=dict(color='white'))
    fig_sankey.show()

## Step 4.4 – NMI Between Consecutive Eras

In [ ]:
ami_results = []
consecutive_pairs = [
    ('cold_war', 'post_cw', 'CW→PCW'),
    ('post_cw',  'post_9_11', 'PCW→9/11'),
    ('post_9_11','recent', '9/11→Recent')
]

for era1, era2, label in consecutive_pairs:
    if era1 not in era_partitions or era2 not in era_partitions:
        print(f'Skipping {label}: one era missing')
        continue
    p1 = era_partitions[era1]
    p2 = era_partitions[era2]
    common = sorted(set(p1.keys()) & set(p2.keys()))
    if len(common) < 10:
        print(f'{label}: too few common countries ({len(common)})')
        continue
    v1 = [p1[c] for c in common]
    v2 = [p2[c] for c in common]
    ami = adjusted_mutual_info_score(v1, v2)
    ami_results.append({'transition': label, 'AMI': round(ami, 4),
                        'interpretation': 'stable' if ami > 0.5 else ('partial rupture' if ami > 0.2 else 'structural rupture')})
    print(f'{label}: AMI = {ami:.4f}  ({"stable" if ami > 0.5 else "partial rupture" if ami > 0.2 else "structural rupture"})')

ami_df = pd.DataFrame(ami_results)
ami_df.to_csv(os.path.join(TABLES, 'p4_ami.csv'), index=False)
print(ami_df.to_string(index=False) if not ami_df.empty else 'No NMI results')

## Step 4.5 – Louvain on 6 Issue Networks (D1: India Multi-Alignment)

In [ ]:
issue_partitions = {}
india_matrix_rows = []

for issue in ISSUES:
    gname = f'issue_{issue}'
    try:
        G_iss = load_graph(gname)
    except FileNotFoundError:
        print(f'Graph {gname} not found, skipping')
        continue

    if G_iss.number_of_edges() < 3:
        print(f'{gname}: too few edges')
        continue

    partition, Q = run_louvain_best(G_iss, n_runs=10)
    issue_partitions[issue] = partition

    # India's community in this issue network
    india_comm = partition.get('India', None)
    if india_comm is None:
        india_comm = next((v for k, v in partition.items() if 'india' in k.lower()), None)

    # Top-5 co-voters by edge weight
    top_covoters = []
    if India in G_iss and india_comm is not None:
        neighbors = [(n, G_iss[India][n]['weight'])
                     for n in G_iss.neighbors(India)]
        top_covoters = sorted(neighbors, key=lambda x: -x[1])[:5]

    n_communities = len(set(partition.values()))
    india_matrix_rows.append({
        'issue': issue, 'issue_name': ISSUE_NAMES[issue],
        'n_communities': n_communities, 'modularity_Q': round(Q, 4),
        'india_community': india_comm,
        'india_top_covoters': ', '.join([f'{c}({w:.2f})' for c, w in top_covoters])
    })
    print(f'{issue} ({ISSUE_NAMES[issue]}): Q={Q:.4f}, comms={n_communities}, India→C{india_comm}')

india_matrix_df = pd.DataFrame(india_matrix_rows)
india_matrix_df.to_csv(os.path.join(TABLES, 'p4_india_alignment_D1.csv'), index=False)
print('\nIndia Multi-Alignment Matrix (D1):')
print(india_matrix_df[['issue_name','india_community','n_communities','modularity_Q']].to_string(index=False))

In [ ]:
# Fix NameError: define India variable
India = 'India'

## Step 4.6 – Girvan-Newman on Human Rights & Nuclear Networks

In [ ]:
gn_issues = ['hr', 'nu']  # human rights, nuclear

for issue in gn_issues:
    gname = f'issue_{issue}'
    try:
        G_iss = load_graph(gname)
    except FileNotFoundError:
        print(f'{gname} not found')
        continue

    print(f'\nGirvan-Newman on {issue} ({G_iss.number_of_nodes()} nodes, {G_iss.number_of_edges()} edges)')

    # Edge betweenness — find bridge edges
    edge_btw = nx.edge_betweenness_centrality(G_iss, weight='weight', normalized=True)
    top_bridges = sorted(edge_btw.items(), key=lambda x: -x[1])[:10]
    print('Top bridge edges (diplomatic hinges):')
    for (u, v), btw in top_bridges:
        w = G_iss[u][v]['weight']
        print(f'  {u:30s} — {v:30s} | btw={btw:.4f}, w={w:.3f}')

    # Run Girvan-Newman (up to 3 splits)
    from networkx.algorithms.community import girvan_newman
    gn_iter = girvan_newman(G_iss)
    try:
        level1 = next(gn_iter)  # first split
        level2 = next(gn_iter)  # second split
        print(f'  After 1 split: {len(level1)} communities ({[len(c) for c in level1]})')
        print(f'  After 2 splits: {len(level2)} communities ({[len(c) for c in level2]})')
    except StopIteration:
        print('  Girvan-Newman exhausted')

## ✅ Phase 4 Complete
**Deliverables:**
- **D1:** `results/tables/p4_india_alignment_D1.csv` — India Multi-Alignment Matrix
- **D2:** `results/plots/p4_sankey_D2.html` — Alluvial diagram of bloc evolution
- Louvain partitions for all 5 eras, NMI values
- Girvan-Newman bridge edges for HR + Nuclear networks

**→ Proceed to Notebook 05: Robustness & Motifs**